In [ ]:
for name in dir():
    if not name.startswith("_") and name not in ["In", "Out", "get_ipython", "exit", "quit"]:
        del globals()[name]


### sport和tech的tf_top1000丟給GAP
 跑混淆矩陣, 看分類的正確率

In [ ]:
import pandas as pd
data = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\0 Meeting\sport_tech_bbcnews.csv').drop(columns='Unnamed: 0')
# 你把 list 存成 .csv 之後，pandas.read_csv 會把它當成 字串，不會還原回原本的 list。
# 所以
import ast
data['Tokens'] = data['Tokens'].apply(ast.literal_eval)
import numpy as np
np.random.seed(42)
data['filename'] = data.index.map(lambda i: f'{i+1:03}.txt')
data['filename'] = data['filename'].astype(str)

gap_df = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\gap_doc_clusters.csv')
gap_df['filename'] = gap_df['filename'].astype(int).apply(lambda x: f"{x:03}.txt")

merged = pd.merge(data, gap_df, on='filename')
merged.head(3)

### sport 分對506筆doc, tech分對392筆doc, 正確率達0.98

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_true = le.fit_transform(merged['category'])+1
y_cluster = merged['gap_cluster'].astype(int).values

print(confusion_matrix(y_true, y_cluster))
print(classification_report(y_true, y_cluster, target_names=le.classes_))

---
---

### 畫圖

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

token_counts = Counter([t for tokens in data['Tokens'] for t in tokens])
valid_tokens = {t for t, c in token_counts.items() if c>=2}
merged['filteredTokens'] = data['Tokens'].apply(lambda ts: [t for t in ts if t in valid_tokens])
merged['filtered_content'] = merged['filteredTokens'].apply(lambda ts: ' '.join(ts))

In [ ]:
print(f'原本token數: {len(token_counts)}, 扣掉只出現1次的token數: {len(valid_tokens)}')

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(merged['filtered_content']).toarray()
tsne = TSNE(n_components=2, random_state=42)
X_embedded = tsne.fit_transform(X)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=merged, x=X_embedded[:, 0], y=X_embedded[:, 1], hue='gap_cluster', palette='tab10')
plt.title('t_SNE')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

---
---

### 用LogisticRegression來學習, 哪些token是重要的

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(merged['filtered_content'])
y = merged['gap_cluster']


clf = LogisticRegression(max_iter=1000)
model_clf = clf.fit(X, y)

feature_names = vectorizer.get_feature_names_out()
coef = model_clf.coef_[0]

top_pos = np.argsort(coef)[-30:]
top_neg = np.argsort(coef)[:30]

print("Cluster 1 特有詞：")
print(feature_names[top_pos])

print("\nCluster 2 特有詞：")
print(feature_names[top_neg])


---
---

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import LdaModel

def train_lda_for_cluster(cluster_id, num_topics=3, topn=30):
    tokens = merged[merged['gap_cluster'] == cluster_id]['filteredTokens'].tolist()
    dictionary = Dictionary(tokens)
    corpus = [dictionary.doc2bow(doc) for doc in tokens]
    lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, passes=10, random_state=42, iterations=100)
    print(f"\n🟦 GAP Cluster {cluster_id} - Top {num_topics} Topics:")
    for i, topic in lda.print_topics(num_topics=num_topics, num_words=topn):
        print(f"Topic {i}: {topic}")
    return lda

lda_cluster_1 = train_lda_for_cluster(1)
lda_cluster_2 = train_lda_for_cluster(2)

🟨 新文件中的詞彙必須在原本的詞表中出現過
否則那部分詞會被忽略，向量稀疏，模型效果變差

若新文章出現很多從未見過的詞 → 模型會表現不佳

所以建議：詞表越大越穩定，但也越容易 overfit

🟨 模型只能預測原本訓練時學到的分類（如 cluster 1&2）
如果新來的文章語意和訓練資料完全不同，則分類可能錯誤，但這是分類模型的自然限制。

---
---

## Banknote資料

In [52]:
import pandas as pd

banknote = pd.read_csv(r"C:\Users\No\Downloads\banknote+authentication\banknote.txt", header=0, names=['variance', 'skewness', 'curtosis', 'entropy', 'class'])
banknote['filename'] = [f'{i}' for i in range(1, 1372)]
banknote['filename'] = banknote['filename'].astype(int)
# banknote = banknote.drop(columns=['class']) 
# banknote.to_csv('banknote.csv', sep='\t', encoding='utf-8', index=False) 
gap_banknote_order = pd.read_csv(r"C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\gap_banknote_order.txt", sep='\t')
raw='1  0  1  0  1  1  0  1  1  1  0  1  1  1  0  1  1  0  1  1  0  1  1  1  0  0  1  1  0  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  0  1  0  1  1  0  1  1  0  1  1  1  0  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  0  1  0  1  1  1  1  0  1  0  1  0  1  1  0  0  1  0  1  1  1  1  1  1  0  1  1  1  1  1  1  1  0  0  1  1  1  0  1  1  1  0  1  1  0  1  1  1  1  1  1  1  1  1  0  0  1  0  1  0  1  0  1  1  1  1  1  1  1  0  0  0  0  1  1  1  1  1  1  0  1  1  1  1  0  0  0  1  1  0  1  1  0  1  1  0  1  1  1  1  0  1  0  0  0  1  0  1  1  1  1  1  0  1  1  0  1  1  1  1  1  1  1  0  1  1  1  0  1  1  1  0  1  1  1  1  0  1  1  1  0  1  0  1  0  1  0  0  1  1  1  0  1  1  1  0  1  1  1  0  0  0  1  1  0  1  1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  0  1  1  0  0  0  0  1  0  0  1  1  1  1  1  1  0  0  1  1  1  0  1  0  1  1  1  1  1  1  0  1  1  0  0  1  1  1  0  1  1  0  1  0  1  0  1  1  1  0  1  1  1  1  0  1  1  1  0  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  0  0  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  0  1  1  0  1  1  1  1  1  0  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  0  0  1  0  1  1  1  1  1  1  0  1  1  0  1  0  1  0  0  0  1  0  1  1  1  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  0  1  1  1  0  0  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  0  1  0  1  1  1  1  1  1  1  1  1  1  0  1  1  0  1  0  0  0  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  0  0  1  1  1  1  1  1  1  1  1  1  1  0  0  0  0  1  0  1  1  1  1  0  1  0  1  0  1  1  1  0  0  1  1  1  0  0  1  0  1  1  0  1  1  1  1  0  1  0  1  1  1  1  1  0  0  1  1  1  1  0  1  0  1  1  1  1  0  0  1  0  1  1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  1  1  0  1  1  1  1  1  1  1  0  0  1  1  1  0  1  0  1  0  1  0  0  0  1  0  0  1  1  0  1  1  1  1  0  1  1  1  1  0  1  0  0  0  1  1  1  1  0  1  0  1  1  0  1  1  0  0  0  1  0  0  0  1  1  1  1  1  1  1  1  0  0  0  1  1  1  1  1  0  1  0  1  1  0  1  1  1  0  1  1  1  0  1  1  0  1  0  1  1  0  0  1  1  1  1  0  1  1  0  1  0  1  1  1  0  1  1  1  0  1  1  1  0  1  1  1  1  1  0  1  0  1  1  1  1  1  1  0  0  0  1  1  0  0  1  1  1  1  1  1  1  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  1  1  1  1  1  0  0  0  1  1  0  0  0  0  1  0  0  0  0  0  1  1  1  0  0  0  0  1  1  1  0  0  0  0  0  1  1  0  0  0  0  1  1  1  0  0  0  1  1  1  1  1  0  0  1  1  1  1  1  1  0  0  1  1  1  1  0  0  0  1  1  0  0  0  0'
raw_list = [int(i) for i in raw.split()]
gap_banknote_order['gap_cluster'] = raw_list
gap_banknote_order['filename'] = gap_banknote_order['UNIQID'].str.lstrip('r').astype('int')+1
gap_banknote_order = gap_banknote_order.drop(columns=['UNIQID'])

banknote_merged = pd.merge(banknote, gap_banknote_order[['gap_cluster', 'filename']], on='filename')
banknote_merged

,variance,skewness,curtosis,entropy,class,filename,gap_cluster
0,4.54590,8.16740,-2.4586,-1.46210,0,1,0
1,3.86600,-2.63830,1.9242,0.10645,0,2,1
2,3.45660,9.52280,-4.0112,-3.59440,0,3,0
3,0.32924,-4.45520,4.5718,-0.98880,0,4,0
4,4.36840,9.67180,-3.9606,-3.16250,0,5,1
...,...,...,...,...,...,...,...
1366,0.40614,1.34920,-1.4501,-0.55949,1,1367,1
1367,-1.38870,-4.87730,6.4774,0.34179,1,1368,1
1368,-3.75030,-13.45860,17.5932,-2.77710,1,1369,1
1369,-3.56370,-8.38270,12.3930,-1.28230,1,1370,1


### clusters評估

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(banknote_merged['class'], banknote_merged['gap_cluster'])
nmi = normalized_mutual_info_score(banknote_merged['class'], banknote_merged['gap_cluster'])

print("ARI:", ari)
print("NMI:", nmi)

### 重新命名GAP的clusters

In [53]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix

# true_y: 原始標籤 (字串或整數均可)
# gap_y : GAP 的 cluster label
conf = confusion_matrix(banknote_merged['class'], banknote_merged['gap_cluster'])
# Hungarian algorithm 找到最大化對角線和的對應
row_ind, col_ind = linear_sum_assignment(-conf)
mapping = {col: row for row, col in zip(row_ind, col_ind)}

# 重新命名 GAP 標籤，之後就跟真實標籤一致
banknote_merged['gap_cluster_aligned'] = np.vectorize(mapping.get)(banknote_merged['gap_cluster'])
acc = (banknote_merged['gap_cluster_aligned'] == banknote_merged['class']).mean()
print(f"GAP 分群與真實標籤對齊後的準確率：{acc:.4f}")


GAP 分群與真實標籤對齊後的準確率：0.5011


### 隨機森林模型評估 baseline

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = banknote_merged.drop(columns=['class', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = banknote_merged['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

banknote_baseline_model = RandomForestClassifier(random_state=42)
banknote_baseline_model.fit(X_train, y_train)
baseline_pred = banknote_baseline_model.predict(X_test)

print("=== Baseline 模型效能 ===")
print(classification_report(y_test, baseline_pred))


### 隨機森林模型評估 gap clusters

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = banknote_merged.drop(columns=['class', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = banknote_merged['gap_cluster']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

banknote_gap_model = RandomForestClassifier(random_state=42)
banknote_gap_model.fit(X_train, y_train)
baseline_pred = banknote_gap_model.predict(X_test)

print("=== banknote_gap_model 模型效能 ===")
print(classification_report(y_test, baseline_pred))


### 隨機森林模型評估 gap 重新命名cluters

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = banknote_merged.drop(columns=['class', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = banknote_merged['gap_cluster_aligned']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

banknote_gap_model = RandomForestClassifier(random_state=42)
banknote_gap_model.fit(X_train, y_train)
baseline_pred = banknote_gap_model.predict(X_test)

print("=== banknote_gap_aligned_model 模型效能 ===")
print(classification_report(y_test, baseline_pred))


### 隨機森林模型評估 弱集成強

In [56]:
banknote_grouped = banknote_merged.groupby('gap_cluster')

X = banknote_merged.drop(columns=['class', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = banknote_merged['gap_cluster_aligned']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

from sklearn.ensemble import RandomForestClassifier
banknote_cluster_models={}
for c, sub_iris in banknote_grouped:
    X_sub = sub_iris.drop(columns=['class', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
    y_sub = sub_iris['class']
    rfc = RandomForestClassifier(random_state=42)
    model = rfc.fit(X_sub, y_sub)
    banknote_cluster_models[c] = model

banknote_router_model = RandomForestClassifier(random_state=42).fit(X_train, banknote_merged.loc[X_train.index, 'gap_cluster'])

def predict(x_new):
    cluster_id = banknote_router_model.predict(pd.DataFrame([x_new], columns=X_train.columns))[0]
    model = banknote_cluster_models[cluster_id]
    return model.predict(pd.DataFrame([x_new], columns=X_train.columns))[0]

banknote_ensemble_preds = [predict(row) for _, row in X_test.iterrows()]

from sklearn.metrics import classification_report

print("=== GAP 集成模型效能 ===")
print(classification_report(y_test, banknote_ensemble_preds))

=== GAP 集成模型效能 ===
              precision    recall  f1-score   support

           0       0.58      0.57      0.57        86
           1       0.30      0.31      0.30        52

    accuracy                           0.47       138
   macro avg       0.44      0.44      0.44       138
weighted avg       0.47      0.47      0.47       138



---
---

### Redwine資料

---
---

## Iris資料

In [ ]:
import pandas as pd
iris = pd.read_csv(r"C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\iris.csv", sep='\t')

iris['filename'] = [f'{i}' for i in range(1, 151)]
iris['filename'] = iris['filename'].astype(int)

gap_iris_order = pd.read_csv(r"C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\gap_iris_order.txt", sep='\t')
values = [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
          2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
          0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1]
gap_iris_order['gap_cluster'] = values
gap_iris_order['filename'] = gap_iris_order['UNIQID'].str.lstrip('r').astype('int')+1
gap_iris_order = gap_iris_order.drop(columns=['UNIQID'])

iris_merged = pd.merge(iris, gap_iris_order[['gap_cluster', 'filename']], on='filename')

iris_merged['Species'] = iris_merged['Species'].replace({'setosa':0, 'versicolor':1, 'virginica':2})

### clusters 評估

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(iris_merged['Species'], iris_merged['gap_cluster'])
nmi = normalized_mutual_info_score(iris_merged['Species'], iris_merged['gap_cluster'])

print("ARI:", ari)
print("NMI:", nmi)

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix

# true_y: 原始標籤 (字串或整數均可)
# gap_y : GAP 的 cluster label
conf = confusion_matrix(iris_merged['Species'], iris_merged['gap_cluster'])
# Hungarian algorithm 找到最大化對角線和的對應
row_ind, col_ind = linear_sum_assignment(-conf)
mapping = {col: row for row, col in zip(row_ind, col_ind)}

# 重新命名 GAP 標籤，之後就跟真實標籤一致
iris_merged['gap_cluster_aligned'] = np.vectorize(mapping.get)(iris_merged['gap_cluster'])


### 隨機森林模型評估 baseline

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = iris_merged.drop(columns=['Species', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = iris_merged['Species']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

iris_baseline_model = RandomForestClassifier(random_state=42)
iris_baseline_model.fit(X_train, y_train)
iris_baseline_pred = iris_baseline_model.predict(X_test)

print("=== Baseline 模型效能 ===")
print(classification_report(y_test, iris_baseline_pred))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = iris_merged.drop(columns=['Species', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = iris_merged['gap_cluster']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

iris_gap_model = RandomForestClassifier(random_state=42)
iris_gap_model.fit(X_train, y_train)
iris_gap_pred = iris_gap_model.predict(X_test)

print("=== gap_model 模型效能 ===")
print(classification_report(y_test, iris_gap_pred))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 假設 'y' 是類別標籤，'gap_cluster' 是 GAP 分群結果
X = iris_merged.drop(columns=['Species', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
y = iris_merged['gap_cluster_aligned']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)

iris_gap_aligned_model = RandomForestClassifier(random_state=42)
iris_gap_aligned_model.fit(X_train, y_train)
iris_gap_aligned_pred = iris_gap_aligned_model.predict(X_test)

print("=== gap_aligned_model 模型效能 ===")
print(classification_report(y_test, iris_gap_aligned_pred))


### 隨機森林模型評估 弱集成強

In [ ]:
iris_grouped = iris_merged.groupby('gap_cluster')

from sklearn.ensemble import RandomForestClassifier
iris_cluster_models={}
for c, sub_iris in iris_grouped:
    X_sub = sub_iris.drop(columns=['Species', 'filename', 'gap_cluster', 'gap_cluster_aligned'])
    y_sub = sub_iris['Species']
    rfc = RandomForestClassifier(random_state=42)
    model = rfc.fit(X_sub, y_sub)
    iris_cluster_models[c] = model

iris_router_model = RandomForestClassifier(random_state=42).fit(X_train, iris_merged.loc[X_train.index, 'gap_cluster'])

def predict(x_new):
    cluster_id = iris_router_model.predict(pd.DataFrame([x_new], columns=X_train.columns))[0]
    model = iris_cluster_models[cluster_id]
    return model.predict(pd.DataFrame([x_new], columns=X_train.columns))[0]

iris_ensemble_preds = [predict(row) for _, row in X_test.iterrows()]

print("=== GAP 集成模型效能 ===")
print(classification_report(y_test, iris_ensemble_preds))

---
---